# 06 · The agent loop

**AI Fundamentals in 3 Hours** · Data Sense

Notebook 05 ended one step short: the model asked for a second tool and nobody was listening.

The fix is a **loop**. That is genuinely the whole architecture of an agent:

> **An agent is a while loop around a tool call.**

In this notebook:

1. Write the agent loop by hand, twelve lines, no framework
2. Replace it with `create_agent` and watch every step
3. Add the guardrail that stops a runaway loop
4. Give it memory
5. Put a human approval gate in front of a destructive tool
6. Assemble everything from the whole workshop into one working agent


In [1]:
# --- run this first, in every notebook ---
import os, json
from pathlib import Path

# read keys out of .env (works from the repo root or from notebooks/)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        for line in candidate.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1-mini"          # provider:model - change this one string to switch providers
model = init_chat_model(MODEL, temperature=0)

import textwrap
def wrap(text, width=88):
    """Print long text wrapped, so answers stay readable on a projector."""
    print(textwrap.fill(str(text), width=width))

assert os.environ.get("OPENAI_API_KEY"), "No API key found - check your .env file"
print("ready |", MODEL)

ready | openai:gpt-4.1-mini


## 1. Setup: tools from notebook 05

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

ORDERS = {
    "48213": {"status": "shipped",    "item": "Table lamp",   "total_inr": 2499,
              "courier": "Delhivery", "eta": "2026-09-16", "placed": "2026-09-08"},
    "91204": {"status": "delivered",  "item": "Storage bins", "total_inr": 4150,
              "courier": "BlueDart",  "eta": "2026-09-10", "placed": "2026-09-05"},
    "77001": {"status": "processing", "item": "Cotton throw", "total_inr": 1280,
              "courier": None,        "eta": None,          "placed": "2026-09-11"},
}

@tool
def get_order_status(order_id: str) -> dict:
    """Look up the live status, item, total and delivery ETA of a customer order.

    Use whenever the user asks where their order is or what they bought.

    Args:
        order_id: The numeric order id, for example 48213
    """
    return ORDERS.get(order_id.strip(), {"error": f"No order found with id {order_id}"}) | {"order_id": order_id}


@tool
def days_until(date_iso: str) -> dict:
    """Calculate how many whole days from today until a given date.

    Use for any "how many days" question. Do NOT do date arithmetic yourself.

    Args:
        date_iso: Target date as YYYY-MM-DD
    """
    from datetime import date
    try:
        target = date.fromisoformat(date_iso)
    except ValueError:
        # Return the error as DATA. Raising here would kill the whole agent run;
        # handing it back lets the model notice and correct itself next turn.
        return {"error": f"{date_iso!r} is not a date. Expected YYYY-MM-DD."}
    return {"date": date_iso, "days_from_today": (target - date.today()).days}


TOOLS = [get_order_status, days_until]
BY_NAME = {t.name: t for t in TOOLS}
print("tools:", list(BY_NAME))

## 2. The agent loop, written by hand

Read this until it is boring. It is the thing every agent framework is built around.

Three rules:

- **If the model asked for tools** → run them, append the results, go around again
- **If the model answered instead** → we are done, return it
- **Always cap the iterations** → otherwise a confused model burns your budget

In [ ]:
def agent_loop(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    model_with_tools = model.bind_tools(TOOLS)
    messages = [
        SystemMessage("You are a support agent for Nimbus Retail. Be concise."),
        HumanMessage(question),
    ]

    for step in range(max_steps):                          # <-- the guardrail
        reply = model_with_tools.invoke(messages)
        messages.append(reply)

        if not reply.tool_calls:                           # <-- it answered, we are done
            if verbose:
                print(f"  step {step}: answered")
            return reply.content

        for call in reply.tool_calls:                      # <-- it asked, so we act
            result = BY_NAME[call["name"]].invoke(call)
            if verbose:
                print(f"  step {step}: {call['name']}({call['args']})")
                print(f"          -> {result.content[:60]}")
            messages.append(result)

    return "Stopped: hit the step limit without finishing."


wrap(agent_loop("Order 48213 - how many days until it arrives?"))

**Two tools, chained, automatically.** It looked up the order, read the ETA out of the
result, then called `days_until` with a date it did not know when it started.

Nobody wrote that plan. Nothing routed it. That emergent chaining is what makes an agent an
agent, and it is why they are harder to predict than a fixed pipeline.

## 3. Now the one-line version

That loop is correct, and you should never write it again. `create_agent` is the same loop,
plus streaming, retries, memory, interrupts and tracing.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt="You are a support agent for Nimbus Retail. Be concise.",
)

result = agent.invoke({"messages": [HumanMessage("Order 48213 - how many days until it arrives?")]})
wrap(result["messages"][-1].content)

### Watch every step

`result["messages"]` is the entire trace. Printing it is the single most useful debugging
habit you can build.

In [ ]:
def print_trace(messages):
    for m in messages:
        label = type(m).__name__
        if getattr(m, "tool_calls", None):
            for c in m.tool_calls:
                print(f"  {label:<14}| CALL {c['name']}({c['args']})")
        else:
            # collapse newlines so a multi-line tool result cannot break the table
            body = " ".join(str(m.content).split())
            print(f"  {label:<14}| {body[:72]}")

print_trace(result["messages"])

Read it top to bottom: question → tool call → result → tool call → result → answer.

That is exactly the hand-written loop, and exactly the diagram from the slides.

### Streaming it live

In a real product you do not want to stare at a blank screen while the agent works.
`stream` gives you each step as it happens.

In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage("What did I order in 91204, and how many days ago was it placed?")]},
    stream_mode="values",
):
    last = chunk["messages"][-1]
    label = type(last).__name__
    if getattr(last, "tool_calls", None):
        print(f"{label:<14}| CALL {[c['name'] for c in last.tool_calls]}")
    else:
        print(f"{label:<14}| {str(last.content)[:74]}")

> **Watch what just happened.** The model may well call `days_until` with the *order id*
> instead of a date. Our tool returns `{"error": ...}` rather than raising, so the agent
> reads the error, recovers, and calls it again with the real date.
>
> If that tool had raised a `ValueError`, the entire run would have died. This is the
> **return errors, don't raise them** rule from notebook 05, and this is why it matters.


## 4. The guardrail

An agent with a badly-described tool can loop forever, and every iteration costs money.
**Never ship an unbounded loop.** In LangChain the cap is `recursion_limit`.

In [ ]:
from langgraph.errors import GraphRecursionError

@tool
def check_warehouse(order_id: str) -> dict:
    """Check whether an order has been picked in the warehouse.

    If it reports not ready, call this tool again to re-check.

    Args:
        order_id: The numeric order id
    """
    return {"ready": False, "note": "Not picked yet. Call check_warehouse again to re-check."}


confused = create_agent(
    model=model,
    tools=[check_warehouse],
    system_prompt=(
        "You are a warehouse bot. You must keep calling check_warehouse until it reports "
        "ready is true. Never answer the user until ready is true."
    ),
)

try:
    confused.invoke(
        {"messages": [HumanMessage("Is order 48213 picked yet?")]},
        config={"recursion_limit": 8},        # <-- stops the bleeding
    )
    print("finished within the limit")
except GraphRecursionError:
    print("GraphRecursionError raised - the cap did its job.")
    print("Without it, this agent would call that tool until your budget ran out.")

That is a deliberately silly system prompt, but the failure is real: a tool that keeps
returning something the model finds unsatisfying will loop until something stops it.

Set `recursion_limit` on every agent you ship.

## 5. Memory

Same checkpointer as notebook 01. The agent now remembers across calls, per `thread_id`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

remembering = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt="You are a support agent for Nimbus Retail. Be concise.",
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "priya"}}

def chat(text: str) -> str:
    out = remembering.invoke({"messages": [HumanMessage(text)]}, config=cfg)
    return out["messages"][-1].content

wrap(chat("Hi, I'm Priya. Can you check order 77001?"))
print()
wrap(chat("Thanks. And what was the item called again?"))       # no second lookup needed

Notice it answered the follow-up **without calling the tool again:** the answer was
already in its conversation history. That is memory saving you a round trip and a few paise.

## 6. A human gate on destructive tools

`get_order_status` only reads. A `cancel_order` tool **destroys data**, and you do not want
a probabilistic system doing that unsupervised.

The split to hold on to: **reads can run freely; anything that writes, sends, pays or
deletes goes behind approval.**

In [ ]:
@tool
def cancel_order(order_id: str) -> dict:
    """Permanently cancel a customer order. This cannot be undone.

    Args:
        order_id: The numeric order id to cancel
    """
    if order_id not in ORDERS:
        return {"error": f"No order found with id {order_id}"}
    ORDERS[order_id]["status"] = "cancelled"
    return {"order_id": order_id, "status": "cancelled"}


from langchain.agents.middleware import HumanInTheLoopMiddleware

guarded = create_agent(
    model=model,
    tools=[get_order_status, cancel_order],
    system_prompt="You are a support agent for Nimbus Retail. Be concise.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"cancel_order": True})],   # reads stay free
    checkpointer=InMemorySaver(),                                                 # required for interrupts
)

approval_cfg = {"configurable": {"thread_id": "approval-demo"}}

guarded.invoke({"messages": [HumanMessage("Please cancel order 91204.")]}, config=approval_cfg)

state = guarded.get_state(approval_cfg)
request = state.interrupts[0].value["action_requests"][0]

print("PAUSED - waiting for a human")
print("  tool :", request["name"])
print("  args :", request["args"])
print()
print("order 91204 status is still:", ORDERS["91204"]["status"], " <- nothing happened yet")

The agent has **stopped mid-run**. The tool did not execute. The decision is now yours,
or your support lead's, or whoever your product says it belongs to.

### Rejecting

In [ ]:
from langgraph.types import Command

reject_cfg = {"configurable": {"thread_id": "reject-demo"}}
guarded.invoke({"messages": [HumanMessage("Cancel order 48213 immediately.")]}, config=reject_cfg)

out = guarded.invoke(
    Command(resume={"decisions": [{"type": "reject"}]}),
    config=reject_cfg,
)
wrap(out["messages"][-1].content)
print()
print("order 48213 status:", ORDERS["48213"]["status"], " <- untouched")

### Approving

In [ ]:
out = guarded.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=approval_cfg,                      # the paused run from before
)
wrap(out["messages"][-1].content)
print()
print("order 91204 status:", ORDERS["91204"]["status"], " <- now it actually changed")

In a real product that pause becomes a Slack message, an approval queue, or a button in
your admin panel. The agent waits; a person decides.

> **This is the single most important production pattern in this notebook.** An agent is a
> loop with credentials. Treat it like any other privileged process.

## 7. Putting the whole workshop together

Everything from the last three hours, in one agent: the RAG retriever from notebook 04
becomes just another tool.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

DATA = Path("../data") if Path("../data").exists() else Path("data")
docs = [Document(page_content=p.read_text(), metadata={"source": p.name}) for p in sorted(DATA.glob("*.md"))]
chunks = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=120, separators=["\n## ", "\n\n", "\n", " ", ""]
).split_documents(docs)

retriever = InMemoryVectorStore.from_documents(
    chunks, OpenAIEmbeddings(model="text-embedding-3-small")
).as_retriever(search_kwargs={"k": 3})


@tool
def search_policies(query: str) -> str:
    """Search Nimbus Retail's official policy documents for refunds, shipping, payments
    and accounts. Use for any question about company policy or rules.

    Args:
        query: What to look for, in natural language
    """
    found = retriever.invoke(query)
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in found)


print(f"indexed {len(chunks)} chunks from {len(docs)} documents")

In [ ]:
support_agent = create_agent(
    model=model,
    tools=[search_policies, get_order_status, days_until, cancel_order],
    system_prompt=(
        "You are a support agent for Nimbus Retail.\n"
        "- Use search_policies for any question about company policy. Never guess policy.\n"
        "- Use get_order_status for anything about a specific order.\n"
        "- Cite the [source] label when you answer from a policy document.\n"
        "- If you cannot find something, say so plainly. Be concise."
    ),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"cancel_order": True})],
    checkpointer=InMemorySaver(),
)

final_cfg = {"configurable": {"thread_id": "final-demo"}}

out = support_agent.invoke(
    {"messages": [HumanMessage(
        "My order 48213 hasn't arrived. How many days until it's due, "
        "and what's your policy if it's late?"
    )]},
    config={**final_cfg, "recursion_limit": 12},
)

print_trace(out["messages"])
print()
print("=" * 74)
wrap(out["messages"][-1].content)

Look at what just happened in one request:

- **Notebook 01:** messages and a system prompt
- **Notebook 02:** every token of it is billed and countable
- **Notebook 03:** the tool schemas are the structured-output mechanism
- **Notebook 04:** `search_policies` is your RAG pipeline
- **Notebook 05:** the model asked, your code executed
- **Notebook 06:** a loop kept going until the job was done, with a cap, memory, and a
  human gate on the one tool that can destroy something

That is a real AI application. You built every layer of it yourself this morning.

## 8. One last piece of judgement

Agents are the exciting answer. They are usually the wrong one.

| | Workflow | Agent |
|---|---|---|
| who decides the steps | **you** | **the model** |
| cost & latency | predictable | varies per request |
| failures | a step broke | it took a path you never imagined |
| testing | step by step | needs real evals |

**If you can draw the flowchart, write the flowchart.** Reach for an agent when you
genuinely cannot know the next step until you have seen the last one.

## 9. Your turn

1. **Add a tool.** Give the agent `check_delivery_pincode(pincode)` returning a delivery
   estimate. Ask a question that needs it plus `search_policies` in the same turn.

2. **Make it loop.** Write a tool that always returns `{"error": "try again"}`. Watch the
   agent retry, then watch `recursion_limit` stop it.

3. **Guard more.** Add `issue_refund(order_id, amount_inr)` to the interrupt list. Try
   `{"type": "edit"}` as a resume decision to change the amount before it runs.

4. **Trace it.** Put a free `LANGSMITH_API_KEY` in your `.env`, set `LANGSMITH_TRACING=true`,
   rerun this notebook, and open smith.langchain.com. Every step, every token, every cost, already instrumented, because you used the framework.

---

### Where to go next

- **LangSmith:** evals and tracing. Start here. It is the skill that separates people who ship.
- **LangGraph:** when you need custom control flow, branching, parallel steps, durable state.
- **Deep Agents:** planning, file management, subagents, long-horizon memory.

You learned one interface this morning. All three of those are built on it.


In [ ]:
# your turn - scratch cell
